# Optical Flow
Reference: [Optical Flow, Vikipedia](https://en.wikipedia.org/wiki/Optical_flow)

Optical flow or optic flow is the pattern of apparent motion of objects, surfaces, and edges in a visual scene caused by the relative motion between an observer and a scene.  Optical flow can also be defined as the distribution of apparent velocities of movement of brightness pattern in an image.

The optical flow methods in OpenCV will first take in a given set of points and a frame. THen it will attempt toi find those points in the next frame. Note the user is providing the points to track.

- Lucas-Kanade: computes optical flow for a sparse feature set (only tracks what was told to track).
- Gunner Farneback's algorigthm is used to calculate **dense optical flow** (say all the points in the frame).

In [1]:
# ----------------
# Lucas-Kanade
# ----------------
import numpy as np
import cv2
corner_track_params = dict(maxCorners = 10, qualityLevel = 0.3, minDistance = 7, blockSize = 7)
lk_params = dict(winSize = (200,200), maxLevel = 2, criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
# points to track
prev_points = cv2.goodFeaturesToTrack(prev_gray, mask = None, **corner_track_params)
mask = np.zeros_like(prev_frame)

while True:
    ret, frame = cap.read()
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    next_points , status, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, prev_points, None, **lk_params)
    good_new = next_points[status == 1]
    good_prev = prev_points[status == 1]
    for i , (new, prev) in enumerate(zip(good_new, good_prev)):
        x_new, y_new = new.ravel()
        x_prev, y_prev = prev.ravel()
        mask = cv2.line(mask, (x_new, y_new),  (x_prev, y_prev), (0,255,0), 2)
        frame = cv2.circle(frame, (x_new, y_new), 5, (0,0,255), -1)
    cv2.add(frame, mask)
    cv2.imshow('tracking', frame)
    k = cv2.waitKey(1) & 0xFF
    if k  == 27:
        break
    prev_gray = frame_gray.copy()
    prev_points = good_new.reshape(-1, 1, 2)

cv2.destroyAllWindows()
cap.release()

error: OpenCV(4.12.0) :-1: error: (-5:Bad argument) in function 'line'
> Overload resolution failed:
>  - Can't parse 'pt1'. Sequence item with index 0 has a wrong type
>  - Can't parse 'pt1'. Sequence item with index 0 has a wrong type
